In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from scikeras.wrappers import KerasRegressor
import torch
import torch.nn as nn
import torch.optim as optim
from skorch import NeuralNetRegressor
from skorch.callbacks import EarlyStopping
from skorch import NeuralNetClassifier

In [13]:
df = pd.read_csv('archive3/diamonds.csv')

categorical_cols = ['cut', 'color', 'clarity']
le = LabelEncoder()
for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

X = df.drop('price', axis=1)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Размер обучающей выборки: {X_train_scaled.shape[0]}")
print(f"Размер тестовой выборки: {X_test_scaled.shape[0]}")

Размер обучающей выборки: 43152
Размер тестовой выборки: 10788


In [18]:
def create_model_tf(neurons1=128, neurons2=64, dropout_rate=0.2, learning_rate=0.001):
    model = Sequential([
        Dense(neurons1, activation='relu', input_shape=(X_train_scaled.shape[1],)),
        Dropout(dropout_rate),
        Dense(neurons2, activation='relu'),
        Dropout(dropout_rate),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    return model

model_tf = KerasRegressor(model=create_model_tf, verbose=2)

param_grid_tf = {
    'model__neurons1': [64, 128],
    'model__neurons2': [32, 64],
    'model__dropout_rate': [0.1, 0.2],
    'model__learning_rate': [0.001, 0.0005],
    'batch_size': [16, 32],
    'epochs': [30]
}

grid_tf = GridSearchCV(
    estimator=model_tf,
    param_grid=param_grid_tf,
    scoring='neg_mean_absolute_error',
    cv=3,
    n_jobs=-1,
    verbose=2
)

print("Запуск GridSearch для TensorFlow...")
grid_tf.fit(X_train_scaled, y_train)

print("\nЛучшие параметры (TF):", grid_tf.best_params_)
print("Лучший MAE (CV):", -grid_tf.best_score_)

Запуск GridSearch для TensorFlow...
Fitting 3 folds for each of 32 candidates, totalling 96 fits


c:\Users\ASUS\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/30
2697/2697 - 6s - 2ms/step - loss: 3672808.2500 - mae: 985.0417
Epoch 2/30
2697/2697 - 4s - 1ms/step - loss: 1618558.3750 - mae: 717.9288
Epoch 3/30
2697/2697 - 4s - 1ms/step - loss: 1520468.1250 - mae: 687.6485
Epoch 4/30
2697/2697 - 4s - 1ms/step - loss: 1466895.5000 - mae: 669.3732
Epoch 5/30
2697/2697 - 4s - 1ms/step - loss: 1450237.0000 - mae: 654.5841
Epoch 6/30
2697/2697 - 4s - 1ms/step - loss: 1403498.3750 - mae: 648.4755
Epoch 7/30
2697/2697 - 4s - 1ms/step - loss: 1366835.1250 - mae: 639.4053
Epoch 8/30
2697/2697 - 3s - 1ms/step - loss: 1342784.8750 - mae: 633.4790
Epoch 9/30
2697/2697 - 4s - 1ms/step - loss: 1344809.2500 - mae: 631.3567
Epoch 10/30
2697/2697 - 3s - 1ms/step - loss: 1343815.2500 - mae: 630.2452
Epoch 11/30
2697/2697 - 3s - 1ms/step - loss: 1325024.6250 - mae: 625.6808
Epoch 12/30
2697/2697 - 3s - 1ms/step - loss: 1343826.1250 - mae: 625.6775
Epoch 13/30
2697/2697 - 4s - 1ms/step - loss: 1308843.8750 - mae: 619.6897
Epoch 14/30
2697/2697 - 3s - 1ms/s

In [19]:
best_model_tf = grid_tf.best_estimator_
y_pred_tf = best_model_tf.predict(X_test_scaled).flatten()

mse_tf = mean_squared_error(y_test, y_pred_tf)
mae_tf = mean_absolute_error(y_test, y_pred_tf)
r2_tf = r2_score(y_test, y_pred_tf)

print("\n--- Результаты модели TensorFlow ---")
print(f"Средняя абсолютная ошибка (MAE) на тесте: {mae_tf:.2f}")
print(f"Среднеквадратичная ошибка (MSE) на тесте: {mse_tf:.2f}")
print(f"Коэффициент детерминации (R^2) на тесте: {r2_tf:.4f}")

675/675 - 1s - 1ms/step

--- Результаты модели TensorFlow ---
Средняя абсолютная ошибка (MAE) на тесте: 513.46
Среднеквадратичная ошибка (MSE) на тесте: 946250.12
Коэффициент детерминации (R^2) на тесте: 0.9405


In [32]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)
y_train_float = y_train.values.astype(np.float32).reshape(-1, 1)

print(f"Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")

Train: (43152, 9), Test: (10788, 9)


In [33]:
from skorch.callbacks import EarlyStopping as SkorchEarlyStopping
from torch.utils.data import DataLoader, TensorDataset

class DiamondPriceModel(nn.Module):
    def __init__(self, input_size=9, neurons1=128, neurons2=64, dropout_rate=0.2):
        super().__init__()
        self.fc1 = nn.Linear(input_size, neurons1)
        self.fc2 = nn.Linear(neurons1, neurons2)
        self.fc3 = nn.Linear(neurons2, 32)
        self.fc4 = nn.Linear(32, 1)
        self.dropout = nn.Dropout(dropout_rate)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.relu(self.fc3(x))
        x = self.fc4(x)
        return x

net = NeuralNetRegressor(
    module=DiamondPriceModel,
    module__input_size=X_train_scaled.shape[1],
    module__neurons1=128,
    module__neurons2=64,
    module__dropout_rate=0.2,
    optimizer=optim.Adam,
    optimizer__lr=0.001,
    max_epochs=30,
    batch_size=32,
    callbacks=[SkorchEarlyStopping(patience=5, lower_is_better=True)],
    criterion=nn.MSELoss,
    verbose=0
)

In [34]:
param_grid = {
    'module__neurons1': [64, 128],
    'module__neurons2': [32, 64],
    'module__dropout_rate': [0.1, 0.2],
    'optimizer__lr': [0.001, 0.0005],
    'batch_size': [16, 32],
}

grid = GridSearchCV(
    estimator=net,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=3,
    n_jobs=-1,
    verbose=2
)

print("\nЗапуск GridSearch для PyTorch")
grid.fit(X_train_scaled, y_train_float)

print("\n=== Лучшие гиперпараметры ===")
best_params = grid.best_params_
for key, value in best_params.items():
    print(f"{key}: {value}")
print(f"Лучший MAE (средний по кросс-валидации): {-grid.best_score_:.2f}")


Запуск GridSearch для PyTorch
Fitting 3 folds for each of 32 candidates, totalling 96 fits

=== Лучшие гиперпараметры ===
batch_size: 16
module__dropout_rate: 0.1
module__neurons1: 128
module__neurons2: 64
optimizer__lr: 0.001
Лучший MAE (средний по кросс-валидации): 561.82


In [35]:
neurons1 = best_params['module__neurons1']
neurons2 = best_params['module__neurons2']
dropout = best_params['module__dropout_rate']
lr = best_params['optimizer__lr']
batch_size = best_params['batch_size']

pt_model = DiamondPriceModel(
    input_size=X_train_scaled.shape[1],
    neurons1=neurons1,
    neurons2=neurons2,
    dropout_rate=dropout
)
criterion = nn.MSELoss()
optimizer = optim.Adam(pt_model.parameters(), lr=lr)

X_train_new, X_val, y_train_new, y_val = train_test_split(
    X_train_scaled, y_train_float, test_size=0.2, random_state=42
)

train_dataset = TensorDataset(
    torch.tensor(X_train_new, dtype=torch.float32),
    torch.tensor(y_train_new, dtype=torch.float32)
)
val_dataset = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.float32)
)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

train_losses, val_losses = [], []
epochs = 100
for epoch in range(epochs):
    pt_model.train()
    epoch_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = pt_model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    train_losses.append(epoch_loss / len(train_loader))

    pt_model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            outputs = pt_model(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()
    val_losses.append(val_loss / len(val_loader))

    if epoch > 10 and val_losses[-1] > min(val_losses[:-5]):
        print(f"Ранняя остановка на эпохе {epoch}")
        break

pt_model.eval()
with torch.no_grad():
    y_pred_pt = pt_model(torch.tensor(X_test_scaled, dtype=torch.float32)).numpy().flatten()

mae_pt = mean_absolute_error(y_test, y_pred_pt)
r2_pt = r2_score(y_test, y_pred_pt)

print("=== PyTorch Results ===")
print(f"MAE: {mae_pt:.2f}, R²: {r2_pt:.4f}")

Ранняя остановка на эпохе 15
=== PyTorch Results ===
MAE: 581.46, R²: 0.9242
